# Optuna Hyperparameter Tuning Sanity Check (Colab, free T4)

**Purpose:** prove the Optuna + training + validation-eval loop actually works end
to end, on a small scale, before trusting it to a real Snellius study. This notebook
runs a handful of trials on small subsamples of `nityaak/mtcsd-stance` -- it is
**not** meant to produce a real tuned hyperparameter choice, just to validate the
mechanism (same relationship `03_colab_hyperparameter_playground.ipynb` has to
`finetune_stance.job`: mechanics preview, not the real experiment).

**What's being tuned:** `learning_rate` and `num_train_epochs` only -- both are still
at TRL's defaults in every real run so far, and are the prime suspects behind
Qwen3-4B's mtcsd fine-tune barely beating zero-shot (0.5179 vs 0.5120 macro F1).
Everything else (LoRA rank/alpha, batch size, weight decay, warmup) stays fixed at
the values already used in `finetune_stance.job`.

**What's deliberately NOT here:** early stopping / Optuna pruning. Both need enough
trials to have a meaningful baseline to compare against -- with only a handful of
sanity-check trials there's nothing to prune against yet. That's for the real
Snellius study, once there's a real trial budget.

**Nothing here touches the Hub** -- no dataset or model pushes, matching notebook
03's convention.

**Setup:** Colab menu -> Runtime -> Change runtime type -> **T4 GPU** (free tier).


## 1. Install dependencies

Same pinned versions as notebook 03, plus Optuna + plotly (for the inline visualization cells later -- `optuna.visualization`'s plotting functions return plotly figures).

In [ ]:
%%capture
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install optuna
!pip install plotly

## 2. Constants

`MODEL_NAME` is Qwen3-**4B** (not the 1.7B used in notebook 03) -- confirmed to
exist on HF (`unsloth/Qwen3-4B-unsloth-bnb-4bit`) before writing this. 4B in 4-bit
plus LoRA should fit a free T4's 16GB, but it's a bigger footprint than the 1.7B
notebook -- section 4 below checks actual peak memory, watch it if you change
`TRAIN_SAMPLE_SIZE`/batch size.


In [ ]:
MODEL_NAME = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 512  # same as finetune_stance.job

DATASET_ID = "nityaak/mtcsd-stance"  # train/validation/test splits, see build_mtcsd_dataset.py

# Small on purpose -- this notebook checks the Optuna mechanism works, not real tuning.
TRAIN_SAMPLE_SIZE = 400
VAL_SAMPLE_SIZE = 100
N_TRIALS = 4

## 3. Load the base model + tokenizer

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto: fp16 on this T4, bf16 on Snellius's A100
    load_in_4bit=True,
)

## 4. GPU memory check (before scaling anything up)

Peak memory for loading the base model alone -- a sanity check specific to this
notebook since 4B is a bigger footprint than notebook 03's 1.7B. If this is already
close to the T4's 16GB, `TRAIN_SAMPLE_SIZE`/batch size may need to shrink further
once training starts.


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total_memory = round(gpu_stats.total_memory / 1024**3, 2)
print(f"Peak reserved memory after model load: {used_memory} GB / {total_memory} GB ({gpu_stats.name})")

## 5. Load mtcsd train/validation splits

Pulls directly from `nityaak/mtcsd-stance`'s `train` and `validation` splits (see
`build_mtcsd_dataset.py` -- these are the official MT-CSD splits, recovered from the
source repo, not a random carve-out). Subsampled down to `TRAIN_SAMPLE_SIZE`/
`VAL_SAMPLE_SIZE` for fast sanity-check trials -- the real Snellius study will use
the full pools (10,925 / 2,209).


In [ ]:
from datasets import load_dataset

train_dataset = load_dataset(DATASET_ID, split="train").shuffle(seed=42).select(range(TRAIN_SAMPLE_SIZE))
val_dataset = load_dataset(DATASET_ID, split="validation").shuffle(seed=42).select(range(VAL_SAMPLE_SIZE))

print(f"train: {len(train_dataset)} examples, validation: {len(val_dataset)} examples")
train_dataset[0]

## 6. Split each conversation into prompt/completion

Same reason as notebook 03: `completion_only_loss` needs the prompt/completion
boundary explicit, not a flattened `"text"` string.


In [ ]:
def split_prompt_completion(examples):
    convos = examples["conversations"]
    return {
        "prompt": [convo[:-1] for convo in convos],
        "completion": [convo[-1:] for convo in convos],
    }


train_dataset = train_dataset.map(split_prompt_completion, batched=True)
val_dataset = val_dataset.map(split_prompt_completion, batched=True)

## 7. `build_model()` -- a fresh model + LoRA adapter per trial

Optuna trials can't share trained weights (trial 2 needs to start from the same
untrained base as trial 1, not continue from wherever trial 1 left off) -- so this
loads a completely fresh base model and applies a fresh LoRA adapter every time it's
called. Same LoRA config as `finetune_stance.py`/notebook 03 (`r=16`, `lora_alpha=16`)
-- fixed, not part of this search.


In [ ]:
def build_model():
    m, tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    m = FastLanguageModel.get_peft_model(
        m,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        max_seq_length=MAX_SEQ_LENGTH,
        random_state=47,
    )
    return m, tok

## 8. The Optuna objective function

`trial.suggest_float(..., log=True)` for `learning_rate` -- log scale because good
learning rates typically span orders of magnitude (1e-5 vs 5e-4 is a meaningful
comparison, 1e-5 vs 1.5e-5 mostly isn't). `trial.suggest_int` for
`num_train_epochs` -- kept to 1-2 here since the sanity-check sample is tiny (400
rows); the real study can widen this.

Everything else mirrors `finetune_stance.job` exactly except `packing` -- turned off
here. With only 400 rows and a couple of epochs, packing (which bundles short
examples into fewer, longer sequences) would collapse this down to very few actual
optimizer steps -- fine for a full-scale run, but it would make the eval-loss signal
too noisy to be a useful sanity check at this sample size.

Returns **validation loss** (`trainer.evaluate()`'s `eval_loss`) -- the cheap,
in-loop signal discussed earlier, not the expensive generation-based macro F1. Lower
is better, so the study below uses `direction="minimize"`.


In [ ]:
from trl import SFTConfig, SFTTrainer


def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    num_train_epochs = trial.suggest_int("num_train_epochs", 1, 2)

    model, tokenizer = build_model()

    sft_config = SFTConfig(
        output_dir=f"colab_outputs/trial_{trial.number}",
        per_device_train_batch_size=8,        # same as finetune_stance.job
        gradient_accumulation_steps=2,         # same as finetune_stance.job
        num_train_epochs=num_train_epochs,     # tuned
        learning_rate=learning_rate,           # tuned
        weight_decay=0.01,                     # fixed, same as finetune_stance.job
        warmup_ratio=0.1,                      # fixed, same as finetune_stance.job
        optim="adamw_8bit",                    # same as finetune_stance.job
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        packing=False,                         # off for this small sample, see markdown above
        eval_strategy="epoch",
        logging_steps=1,
        max_length=MAX_SEQ_LENGTH,
        completion_only_loss=True,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    trainer.train()
    eval_result = trainer.evaluate()
    val_loss = eval_result["eval_loss"]

    print(f"Trial {trial.number}: lr={learning_rate:.2e}, epochs={num_train_epochs} -> val_loss={val_loss:.4f}")

    del model, tokenizer, trainer
    torch.cuda.empty_cache()

    return val_loss

## 9. Create the study with persistent storage

`storage="sqlite:///..."` instead of the default in-memory study -- this is what
lets `optuna-dashboard` (or, on Colab, the inline plots in section 11) read the
study's results. `study_name` is fixed so re-running this cell resumes the same
study instead of silently starting a new one.


In [ ]:
import optuna

study = optuna.create_study(
    study_name="qwen3-4b-mtcsd-tuning-colab-sanity",
    storage="sqlite:///mtcsd_tuning_study.db",
    direction="minimize",
    load_if_exists=True,
)

## 10. Run the study

`N_TRIALS = 4` -- enough to confirm the loop works and see Optuna's sampler adjust
its suggestions between trials, small enough to finish in a reasonable time on a
free T4. Each trial fully reloads the model (section 7), so expect this cell to take
roughly `N_TRIALS` times as long as a single training run.


In [ ]:
study.optimize(objective, n_trials=N_TRIALS)

print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best val_loss: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

## 11. Inline visualization (Colab-friendly alternative to the dashboard)

`optuna-dashboard` is a background web server -- awkward inside a Colab session.
`optuna.visualization`'s functions return the same information as plotly figures
that render directly in the notebook. Each is wrapped in a `try`/`except` --
`plot_param_importances` in particular needs at least a couple of trials with
actual variance in their outcomes to compute anything meaningful, and can fail with
too few trials.


In [ ]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

plot_optimization_history(study).show()

In [ ]:
try:
    plot_param_importances(study).show()
except (ValueError, RuntimeError) as e:
    print(f"Skipped param-importance plot: {e}")

In [ ]:
try:
    plot_parallel_coordinate(study).show()
except (ValueError, RuntimeError) as e:
    print(f"Skipped parallel-coordinate plot: {e}")

## Recap

- Confirms the full loop works: Optuna suggests hyperparameters -> fresh model +
  LoRA per trial -> train on a mtcsd subsample -> validation loss computed in-loop
  -> reported back to the study -> next trial's suggestion informed by it.
- Deliberately small (400/100 train/val rows, 4 trials, no pruning) -- a mechanics
  check, not a real tuning result. Don't read anything into `study.best_params` here.
- `mtcsd_tuning_study.db` (written to the Colab runtime's local disk) is what
  `optuna-dashboard` would read -- download it if you want to browse this
  sanity-check study locally, though the real signal will come from the Snellius
  study's own `.db` file.
- Once this mechanism is confirmed working: build the Snellius `.py`/`.job` pair
  running the same `objective()` shape against the **full** `nityaak/mtcsd-stance`
  train (10,925) / validation (2,209) pools, with a real trial budget and pruning
  enabled.
